In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import datetime

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc

import matplotlib.pyplot as plt

In [2]:
# Base directory
save_directory = r"try_05"
path = os.path.join(save_directory , "featured_144days.pkl")

df = pd.read_pickle(path)
print("Shape : " , df.shape)
df.head()

Shape :  (1720181, 35)


,step,amount,initiator,oldBalInitiator,newBalInitiator,recipient,oldBalRecipient,newBalRecipient,isFraud,index,...,INIT_AMOUNT_DEV_TX_24,INIT_TX_COUNT_STEP_6,INIT_TX_COUNT_STEP_12,INIT_TX_COUNT_STEP_24,RECIP_RISK_6,RECIP_TX_COUNT_6,RECIP_RISK_12,RECIP_TX_COUNT_12,RECIP_RISK_24,RECIP_TX_COUNT_24
0,0,19824.96,4537027967639631,187712.18,167887.22,4875702729424478,8.31,19833.27,1,0,...,0.00,1,1,1,0.0,0.0,0.0,0.0,0.0,0.0
1,0,598.97,4296267625767470,8.92,8.92,25-0000401,0.00,0.00,0,1,...,0.00,1,1,1,0.0,0.0,0.0,0.0,0.0,0.0
2,0,545.85,4178224023847746,93.60,-452.25,13-0001587,0.00,545.85,0,2,...,0.00,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0
3,0,19847.01,4178224023847746,-452.25,-20299.26,4096920916696293,4011.72,23858.74,1,3,...,9650.58,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0
4,0,546.89,4779013371563747,159148.76,158601.88,75-0003564,0.00,546.89,0,4,...,0.00,4,4,4,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
df.columns

Index(['step', 'amount', 'initiator', 'oldBalInitiator', 'newBalInitiator',
       'recipient', 'oldBalRecipient', 'newBalRecipient', 'isFraud', 'index',
       'transactionType_DEBIT', 'transactionType_DEPOSIT',
       'transactionType_PAYMENT', 'transactionType_TRANSFER',
       'transactionType_WITHDRAWAL', 'amount_balance_ratio',
       'balance_increase_ratio', 'neg_balance_i', 'balance_error_i',
       'balance_error_r', 'INIT_AVG_AMOUNT_TX_6', 'INIT_AMOUNT_DEV_TX_6',
       'INIT_AVG_AMOUNT_TX_12', 'INIT_AMOUNT_DEV_TX_12',
       'INIT_AVG_AMOUNT_TX_24', 'INIT_AMOUNT_DEV_TX_24',
       'INIT_TX_COUNT_STEP_6', 'INIT_TX_COUNT_STEP_12',
       'INIT_TX_COUNT_STEP_24', 'RECIP_RISK_6', 'RECIP_TX_COUNT_6',
       'RECIP_RISK_12', 'RECIP_TX_COUNT_12', 'RECIP_RISK_24',
       'RECIP_TX_COUNT_24'],
      dtype='object')

In [4]:
df = df.sort_values(["step", "index"]).reset_index(drop=True)

delay = 6

max_step = df["step"].max()

train_end = int(max_step * 0.8)
delay_end = train_end + delay

print("Train ends at step:", train_end)
print("Delay ends at step:", delay_end)

train_df = df[df["step"] <= train_end]
delay_df = df[(df["step"] > train_end) & (df["step"] <= delay_end)]  # not used
test_df = df[df["step"] > delay_end]

Train ends at step: 114
Delay ends at step: 120


In [5]:
drop_cols = [
    "step",
    "recipient",
    "initiator",
    "index"
]

In [6]:
x_train = train_df.drop(columns=drop_cols + ["isFraud"])
y_train = train_df["isFraud"]

x_test = test_df.drop(columns=drop_cols + ["isFraud"])
y_test = test_df["isFraud"]

In [7]:
print(x_train.dtypes)

amount                        float64
oldBalInitiator               float64
newBalInitiator               float64
oldBalRecipient               float64
newBalRecipient               float64
transactionType_DEBIT            bool
transactionType_DEPOSIT          bool
transactionType_PAYMENT          bool
transactionType_TRANSFER         bool
transactionType_WITHDRAWAL       bool
amount_balance_ratio          float64
balance_increase_ratio        float64
neg_balance_i                   int64
balance_error_i               float64
balance_error_r               float64
INIT_AVG_AMOUNT_TX_6          float64
INIT_AMOUNT_DEV_TX_6          float64
INIT_AVG_AMOUNT_TX_12         float64
INIT_AMOUNT_DEV_TX_12         float64
INIT_AVG_AMOUNT_TX_24         float64
INIT_AMOUNT_DEV_TX_24         float64
INIT_TX_COUNT_STEP_6            int64
INIT_TX_COUNT_STEP_12           int64
INIT_TX_COUNT_STEP_24           int64
RECIP_RISK_6                  float64
RECIP_TX_COUNT_6              float64
RECIP_RISK_1

In [8]:
print("Train : ", x_train.shape)
print("Test : ", x_test.shape)
print("\n")
print("Train frauds : ", y_train.sum())
print("Train fraud rate : ", y_train.mean())
print("\n")
print("Test frauds : ", y_test.sum())
print("Test fraud rate : ", y_test.mean())

Train :  (1361460, 30)
Test :  (303058, 30)


Train frauds :  140466
Train fraud rate :  0.10317306421047992


Test frauds :  27938
Test fraud rate :  0.0921869741105663


In [9]:
rf1 = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf1.fit(x_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_jobs=-1,
                       random_state=42)

In [10]:
rf2 = RandomForestClassifier(
    n_estimators=100,
    max_depth=35,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf2.fit(x_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=35, n_jobs=-1,
                       random_state=42)

In [11]:
os.makedirs(save_directory, exist_ok=True)

joblib.dump(rf1, os.path.join(save_directory , "model_default.pkl"))
joblib.dump(rf2, os.path.join(save_directory , "model_maxdepth35.pkl"))

['try_05\\model_maxdepth35.pkl']

In [12]:
x_train.to_pickle(os.path.join(save_directory , "x_train.pkl"))
x_test.to_pickle(os.path.join(save_directory , "x_test.pkl"))

y_train.to_pickle(os.path.join(save_directory , "y_train.pkl"))
y_test.to_pickle(os.path.join(save_directory , "y_test.pkl"))